## 1. Setup

In [ ]:
!pip install -q transformers torch torchvision datasets timm scikit-learn

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import timm
from datasets import load_dataset
from functools import partial
import numpy as np
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 2. Configuration

In [ ]:
CONFIG = {
    'batch_size': 32,
    'num_epochs': 20,
    'learning_rate': 1e-4,
    'image_size': 224,
    'num_workers': 2
}

print("Configuration loaded")

Configuration loaded


## 3. Load Dataset

In [ ]:
print("Loading dataset...")
dataset = load_dataset("irisxue/real-and-fake-video-frames")

print(f"Total frames: {len(dataset['train']):,}")

# Split dataset
dataset_split = dataset['train'].train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split['train']
val_dataset = dataset_split['test']

print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")

Loading dataset...


pika_frames.zip:   0%|          | 0.00/552M [00:00<?, ?B/s]

real_frames_1.zip:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

sora_frames_4.zip:   0%|          | 0.00/554M [00:00<?, ?B/s]

t2vz_frames.zip:   0%|          | 0.00/489M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Total frames: 49,992
Training samples: 39,993
Validation samples: 9,999


## 4. Data Preprocessing

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def collate_fn(batch, transform):
    images = []
    labels = []

    for item in batch:
        try:
            img = item['image']
            if img.mode != 'RGB':
                img = img.convert('RGB')

            img_tensor = transform(img)
            images.append(img_tensor)

            # Fix label - ensure it's 0 or 1
            label = item['label']
            if isinstance(label, (list, tuple)):
                label = label[0]
            label = int(label)
            # Clamp to 0 or 1
            label = min(max(label, 0), 1)
            labels.append(label)
        except Exception as e:
            print(f"Error processing item: {e}")
            continue

    if len(images) == 0:
        return None, None

    return torch.stack(images), torch.LongTensor(labels)

print("Transforms configured")

Transforms configured


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    collate_fn=partial(collate_fn, transform=train_transform),
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    collate_fn=partial(collate_fn, transform=val_transform),
    pin_memory=True
)

print(f"DataLoaders created")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

DataLoaders created
Train batches: 1250
Val batches: 313


## 5. Model Definition

In [ ]:
class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b5', pretrained=True)
        num_features = self.backbone.classifier.in_features

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        return self.backbone(x)

print("Initializing model...")
model = DeepfakeDetector().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Initializing model...


model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

Total parameters: 29,390,898


## 6. Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("Training components initialized")

Training components initialized


## 7. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        if images is None:
            continue

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(loader)}, Loss: {loss.item():.4f}")

    return total_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            if images is None:
                continue

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total

print("Training functions ready")

Training functions ready


In [ ]:
%%bash
# Create extraction directory
mkdir -p /content/extracted_all

# Find and unzip all HuggingFace dataset ZIPs
find ~/.cache/huggingface/datasets/irisxue* -name "*.zip" -exec unzip -q {} -d /content/extracted_all \;

echo "Done! All files extracted to /content/extracted_all"
ls -lh /content/extracted_all

Done! All files extracted to /content/extracted_all
total 0


In [ ]:
%%bash
# Find ALL zip files in huggingface cache
echo "Searching for ZIP files..."
find ~/.cache/huggingface -name "*.zip" 2>/dev/null

echo ""
echo "HuggingFace dataset structure:"
ls -la ~/.cache/huggingface/datasets/

Searching for ZIP files...
/root/.cache/huggingface/hub/datasets--irisxue--real-and-fake-video-frames/snapshots/efed8a50e43b5fa0777c6d546b387a4f99d0bd0c/real_frames_1.zip
/root/.cache/huggingface/hub/datasets--irisxue--real-and-fake-video-frames/snapshots/efed8a50e43b5fa0777c6d546b387a4f99d0bd0c/sora_frames_4.zip
/root/.cache/huggingface/hub/datasets--irisxue--real-and-fake-video-frames/snapshots/efed8a50e43b5fa0777c6d546b387a4f99d0bd0c/pika_frames.zip
/root/.cache/huggingface/hub/datasets--irisxue--real-and-fake-video-frames/snapshots/efed8a50e43b5fa0777c6d546b387a4f99d0bd0c/t2vz_frames.zip

HuggingFace dataset structure:
total 12
drwxr-xr-x 3 root root 4096 Apr  1 08:50 .
drwxr-xr-x 5 root root 4096 Apr  1 08:50 ..
drwxr-xr-x 3 root root 4096 Apr  1 08:50 irisxue___real-and-fake-video-frames


In [ ]:
%%bash
# Create extraction directory
mkdir -p /content/frames_extracted

# Set the base path
BASE_PATH="/root/.cache/huggingface/hub/datasets--irisxue--real-and-fake-video-frames/snapshots/efed8a50e43b5fa0777c6d546b387a4f99d0bd0c"

# Unzip each file
echo "Unzipping t2vz_frames.zip..."
unzip -q "$BASE_PATH/t2vz_frames.zip" -d /content/frames_extracted

echo "Unzipping real_frames_1.zip..."
unzip -q "$BASE_PATH/real_frames_1.zip" -d /content/frames_extracted

echo "Unzipping sora_frames_4.zip..."
unzip -q "$BASE_PATH/sora_frames_4.zip" -d /content/frames_extracted

echo "Unzipping pika_frames.zip..."
unzip -q "$BASE_PATH/pika_frames.zip" -d /content/frames_extracted

echo ""
echo "Done! All frames extracted to /content/frames_extracted"
echo ""
echo "Total frames:"
find /content/frames_extracted -type f | wc -l

Unzipping t2vz_frames.zip...
Unzipping real_frames_1.zip...
Unzipping sora_frames_4.zip...
Unzipping pika_frames.zip...

Done! All frames extracted to /content/frames_extracted

Total frames:
49992


In [ ]:
import os
from pathlib import Path

extracted_path = Path('/content/frames_extracted')

# See what's inside
print("Directory structure:")
for item in sorted(extracted_path.rglob('*'))[:20]:
    print(item)

Directory structure:
/content/frames_extracted/real_frames_1
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_01.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_02.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_03.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_04.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_05.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_06.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_07.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_08.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_09.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_10.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_frame_11.jpg
/content/frames_extracted/real_frames_1/ILSVRC2015_train_00005003_fram

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os

class FastFrameDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples = []

        # Collect all image files and assign labels based on folder/filename
        root = Path(root_dir)

        for img_path in root.rglob('*.jpg'):
            # Determine label from path
            path_str = str(img_path).lower()
            if 'real' in path_str:
                label = 0
            else:  # pika, sora, t2vz are all AI-generated
                label = 1

            self.samples.append((str(img_path), label))

        print(f"Loaded {len(self.samples)} samples")
        real_count = sum(1 for _, label in self.samples if label == 0)
        fake_count = len(self.samples) - real_count
        print(f"  Real: {real_count}, Fake: {fake_count}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# Create datasets
print("Creating fast datasets from extracted files...")
full_dataset = FastFrameDataset('/content/frames_extracted', transform=None)

# Split into train/val
from sklearn.model_selection import train_test_split
train_indices, val_indices = train_test_split(
    range(len(full_dataset)),
    test_size=0.2,
    random_state=42
)

# Create train/val datasets
train_dataset_fast = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset_fast = torch.utils.data.Subset(full_dataset, val_indices)

# Wrap with transforms
class TransformDataset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img_path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset_fast = TransformDataset(train_dataset_fast, train_transform)
val_dataset_fast = TransformDataset(val_dataset_fast, val_transform)

print(f"\nTrain: {len(train_dataset_fast)}, Val: {len(val_dataset_fast)}")

Creating fast datasets from extracted files...
Loaded 49992 samples
  Real: 20000, Fake: 29992

Train: 39993, Val: 9999


In [ ]:
# Create new fast dataloaders
train_loader = DataLoader(
    train_dataset_fast,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset_fast,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Fast DataLoaders ready!")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print("\nNow training will be FAST - 1-2 minutes per epoch instead of 30!")

Fast DataLoaders ready!
Train batches: 1250
Val batches: 313

Now training will be FAST - 1-2 minutes per epoch instead of 30!


In [ ]:
print(f"Starting training for {CONFIG['num_epochs']} epochs")
print()

best_val_acc = 0
patience_counter = 0
patience = 5

for epoch in range(CONFIG['num_epochs']):
    print(f"Epoch {epoch+1}/{CONFIG['num_epochs']}")

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    print(f"  Training - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")

    val_loss, val_acc = validate(model, val_loader, criterion, device)
    print(f"  Validation - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  Best model saved! Val Acc: {val_acc:.2f}%")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

    print()

print(f"Training completed!")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

Starting training for 20 epochs

Epoch 1/20
  Batch 0/1250, Loss: 0.6670
  Batch 100/1250, Loss: 0.2015
  Batch 200/1250, Loss: 0.4310
  Batch 300/1250, Loss: 0.0072
  Batch 400/1250, Loss: 0.0066
  Batch 500/1250, Loss: 0.0381
  Batch 600/1250, Loss: 0.0197


## 8. Evaluation

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

val_loss, val_acc = validate(model, val_loader, criterion, device)

print("Final Model Performance")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.2f}%")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        if images is None:
            continue
        images = images.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['Real', 'AI-Generated']))

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(f"              Predicted")
print(f"              Real  Fake")
print(f"Actual Real   {cm[0][0]:5d} {cm[0][1]:5d}")
print(f"Actual Fake   {cm[1][0]:5d} {cm[1][1]:5d}")

## 9. Save Model

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'val_acc': best_val_acc,
    'config': CONFIG
}, 'genvscan_final.pth')

print(f"Model saved: genvscan_final.pth")
print(f"Best accuracy: {best_val_acc:.2f}%")

from google.colab import files
files.download('genvscan_final.pth')

In [ ]:
model = spectra()
model.load_state_dict(genvscan_final.pth)

## 10. Test

In [ ]:
import cv2
import numpy as np

def predict_video(video_path, model, transform, device, num_frames=16):
    model.eval()
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        return None

    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    predictions = []
    probabilities = []

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_pil = Image.fromarray(frame_rgb)
        img_tensor = transform(frame_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(img_tensor)
            probs = torch.softmax(output, dim=1)
            pred = output.argmax(1).item()
            predictions.append(pred)
            probabilities.append(probs[0].cpu().numpy())

    cap.release()

    predictions = np.array(predictions)
    probabilities = np.array(probabilities)
    fake_votes = np.sum(predictions == 1)
    real_votes = np.sum(predictions == 0)
    avg_probs = np.mean(probabilities, axis=0)
    final_pred = 1 if fake_votes > real_votes else 0
    final_label = "AI-Generated" if final_pred == 1 else "Real"

    print(f"\nVIDEO ANALYSIS")
    print(f"Prediction: {final_label}")
    print(f"Confidence: {max(avg_probs)*100:.2f}%")
    print(f"Real: {avg_probs[0]*100:.2f}%, AI-Generated: {avg_probs[1]*100:.2f}%")

    return final_label

# Test it
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
result = predict_video(video_path, model, val_transform, device, num_frames=16)